# P3: Acquisition & Integration
**DSA 405 · Fall 2026 · Project Milestone 3** · Unity ID: `svyas2`

**Question carried over from P1:** how do Wake County restaurant inspection scores (Source 1, open-data CSV) relate to the establishment records published on the county's public health portal (Source 2, scraped from CDP)?

| Deliverable | Where |
|---|---|
| 1. Acquisition code (web source = Source 2, CDP portal) | Section 1 |
| 2. Ethics and legality in practice | Section 2 |
| 3. Join design | Section 3 |
| 4. Verification suite | Section 4 |
| 5. Prove the suite works | Section 5 |
| One surprise | Section 6 |
| Self-scored rubric | Section 7 |



repo link: https://github.com/usersimvy/dsa405-project/tree/main

In [ ]:
import hashlib
import io
import re
import time
from pathlib import Path
from urllib import robotparser
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ---- identity and politeness (matches the A8 plan) ----
UNITY_ID = "svyas2"
USER_AGENT = f"DSA405-Student-Project/1.0 (coursework, NC State University; {UNITY_ID}@ncsu.edu)"
MIN_DELAY_SEC = 2.0     # A8 promised 1 request / 2 s; the P3 minimum is 1 / s, so this is stricter
TIMEOUT_SEC = 20        # a request that hangs longer than this is treated as failed
MAX_RETRIES = 3         # for timeouts, connection errors, 429 and 5xx only
BACKOFF_SEC = 5         # wait BACKOFF_SEC * attempt before retrying

# ---- sources ----
SRC1_URL = "https://data-wake.opendata.arcgis.com/api/download/v1/items/ebe3ae7f76954fad81411612d7c4fb17/csv?layers=1"
SRC2_URL = "https://public.cdpehs.com/NCENVPBL/ESTABLISHMENT/ShowESTABLISHMENTTablePage.aspx?ESTTST_CTY=92"
ROBOTS_URL = "https://public.cdpehs.com/robots.txt"

# ---- scrape scope (see drift table in Section 2) ----
MAX_PAGES = 40          # the full site is ~3,989 pages (~2.2 h at 2 s/page); raise this to widen coverage

# ---- table settings found by the discovery cell ----
DATA_TABLE_ID = "VW_PUBLIC_ESTINSPTableControlGrid"
REQUIRED_COLUMNS = ["Inspection Date", "Premises Name", "Address", "State ID#", "Establishment Type", "Final Score"]
MIN_ROWS_PER_PAGE = 2   # a page with fewer data rows than this is treated as a failed parse

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})
print("User-Agent:", USER_AGENT)

User-Agent: DSA405-Student-Project/1.0 (coursework, NC State University; svyas2@ncsu.edu)


## 1. Acquisition code (Source 2: CDP public health portal)

Requirements covered here: rate limit, honest User-Agent, disk cache, pagination, timeouts / non-200 / empty-selector handling, and loud failure with a demonstrated catch.

**Design notes**
* `fetch()` is the only function that touches the network. It checks the cache first, waits its turn, retries only the failures worth retrying, and raises `AcquisitionError` for everything else. There is no bare `except`.
* The portal is ASP.NET: "next page" is a form postback that must carry the hidden `__VIEWSTATE` fields from the previous page. Page *N+1* is requested using the cached HTML of page *N*, so a re-run rebuilds the whole chain from disk.
* A stuck postback returns the same page again with a 200 status, which would silently duplicate rows. `assert_page_advanced()` catches that.

In [ ]:
class AcquisitionError(RuntimeError):
    """Raised when continuing would silently produce wrong or missing data."""


_last_request_at = 0.0


def _wait_turn():
    """Sleep so that consecutive requests are at least MIN_DELAY_SEC apart."""
    global _last_request_at
    RAW_DIR.mkdir(parents=True, exist_ok=True)     # make sure the cache folder exists before any download is saved
    gap = time.monotonic() - _last_request_at
    if gap < MIN_DELAY_SEC:
        time.sleep(MIN_DELAY_SEC - gap)
    _last_request_at = time.monotonic()


def fetch(url, cache_path, data=None):
    """Return the text of `url`, from disk if cached, otherwise from the network.

    GET when `data` is None, POST (form-encoded) otherwise.
    Retries timeouts, connection errors, 429 and 5xx. Any other non-200 status,
    or a 200 with an empty body, raises AcquisitionError immediately.
    """
    cache_path = Path(cache_path)
    if cache_path.exists():                        # never fetch the same resource twice
        return cache_path.read_text(encoding="utf-8")

    problem = None
    for attempt in range(1, MAX_RETRIES + 1):
        _wait_turn()
        try:
            resp = session.request("POST" if data else "GET", url, data=data, timeout=TIMEOUT_SEC)
        except (requests.Timeout, requests.ConnectionError) as exc:
            problem = f"{type(exc).__name__}: {exc}"
        else:
            if resp.status_code == 200:
                if not resp.text.strip():
                    raise AcquisitionError(f"HTTP 200 but empty body from {url}")
                cache_path.write_text(resp.text, encoding="utf-8")
                return resp.text
            if resp.status_code == 429 or resp.status_code >= 500:
                problem = f"HTTP {resp.status_code}"
            else:
                raise AcquisitionError(f"HTTP {resp.status_code} from {url} (not retryable)")
        print(f"  attempt {attempt}/{MAX_RETRIES} failed ({problem})")
        time.sleep(BACKOFF_SEC * attempt)
    raise AcquisitionError(f"gave up on {url} after {MAX_RETRIES} attempts; last problem: {problem}")

In [ ]:
def check_robots(url):
    """Stop if robots.txt disallows `url` for us or asks for a slower crawl than we use.

    A 404 means the site publishes no robots.txt, so nothing is disallowed; we record that and continue.
    Any other failure to read it stops the run.
    """
    cache = RAW_DIR / "cdp_robots.txt"
    if cache.exists():
        txt = cache.read_text(encoding="utf-8")
    else:
        _wait_turn()
        try:
            resp = session.get(ROBOTS_URL, timeout=TIMEOUT_SEC)
        except (requests.Timeout, requests.ConnectionError) as exc:
            raise AcquisitionError(f"could not reach robots.txt: {exc}") from exc
        if resp.status_code == 404:
            txt = ""                                   # empty file = "no robots.txt published"
        elif resp.status_code == 200:
            txt = resp.text
        else:
            raise AcquisitionError(f"unexpected HTTP {resp.status_code} from {ROBOTS_URL}")
        cache.write_text(txt, encoding="utf-8")

    if not txt.strip():
        print(f"No robots.txt published at {ROBOTS_URL} (HTTP 404): no crawl restrictions stated by the site.")
        return
    rp = robotparser.RobotFileParser()
    rp.parse(txt.splitlines())
    if not rp.can_fetch(USER_AGENT, url):
        raise AcquisitionError(f"robots.txt disallows {url} for our User-Agent")
    delay = rp.crawl_delay(USER_AGENT)
    if delay is not None and delay > MIN_DELAY_SEC:
        raise AcquisitionError(f"robots.txt Crawl-delay {delay}s is slower than our {MIN_DELAY_SEC}s")
    print("robots.txt allows:", url)
    print("robots.txt text, for the record:\n" + txt[:600])


def parse_page(html, page_no):
    """Extract the data table from one page of HTML as a DataFrame.

    Raises AcquisitionError if the selector matches nothing, the table is
    malformed, a required column is missing, or too few rows came back.
    """
    soup = BeautifulSoup(html, "html.parser")
    if DATA_TABLE_ID:
        table = soup.find("table", id=DATA_TABLE_ID)
        if table is None:
            raise AcquisitionError(f"page {page_no}: no <table id='{DATA_TABLE_ID}'> found; has the site changed?")
    else:
        # Layout tables are nested; the data table is the innermost table with the most rows.
        leaf_tables = [t for t in soup.find_all("table") if t.find("table") is None]
        if not leaf_tables:
            raise AcquisitionError(f"page {page_no}: no <table> elements at all; selector matched nothing")
        table = max(leaf_tables, key=lambda t: len(t.find_all("tr")))
    try:
        df = pd.read_html(io.StringIO(str(table)))[0]
    except ValueError as exc:
        raise AcquisitionError(f"page {page_no}: table could not be parsed ({exc})") from exc
    df = df.dropna(how="all")
    if len(df) < MIN_ROWS_PER_PAGE:
        raise AcquisitionError(f"page {page_no}: only {len(df)} data rows (minimum {MIN_ROWS_PER_PAGE})")
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise AcquisitionError(f"page {page_no}: missing expected columns {missing}; got {list(df.columns)}")
    return df.assign(_page=page_no)


def assert_page_advanced(df, seen_hashes, page_no):
    """A stuck ASP.NET postback re-serves the same page with status 200. Catch that."""
    digest = hashlib.sha256(df.drop(columns="_page").to_csv(index=False).encode()).hexdigest()
    if digest in seen_hashes:
        raise AcquisitionError(f"page {page_no} has identical content to an earlier page; pagination is stuck")
    seen_hashes.add(digest)


def find_next_control(soup):
    """Return the enabled 'next page' link or button, or None on the last page."""
    for el in soup.find_all(["a", "input"]):
        label = " ".join(filter(None, [el.get("id"), el.get("name"), el.get("title"),
                                       el.get("alt"), el.get("value"), el.get_text(" ", strip=True)])).lower()
        if "next" in label and not el.has_attr("disabled") and el.get("aria-disabled") != "true":
            return el
    return None


def build_postback(html, control):
    """Build the form payload that 'clicks' `control`, carrying all hidden ASP.NET state."""
    soup = BeautifulSoup(html, "html.parser")
    form = soup.find("form")
    if form is None:
        raise AcquisitionError("no <form> on page; cannot build ASP.NET postback")
    payload = {i["name"]: i.get("value", "") for i in form.find_all("input", {"type": "hidden"}) if i.get("name")}
    if "__VIEWSTATE" not in payload:
        raise AcquisitionError("__VIEWSTATE missing; the page structure has changed")
    if control.name == "input":
        name = control.get("name")
        payload[name] = control.get("value", "")
        if control.get("type") == "image":
            payload[name + ".x"], payload[name + ".y"] = "1", "1"
    else:
        m = re.search(r"__doPostBack\('([^']*)','([^']*)'", control.get("href", ""))
        if not m:
            raise AcquisitionError(f"next link has no __doPostBack target: {control.get('href')!r}")
        payload["__EVENTTARGET"], payload["__EVENTARGUMENT"] = m.groups()
    return payload, urljoin(SRC2_URL, form.get("action") or "")

### Discovery cell
This fetches page 1 (cached afterwards) and prints every innermost table and the detected "next" control. It is how `DATA_TABLE_ID` and `REQUIRED_COLUMNS` in the config cell were chosen from evidence rather than guessed.

In [ ]:
check_robots(SRC2_URL)
page1_html = fetch(SRC2_URL, RAW_DIR / "cdp_page_0001.html")
soup1 = BeautifulSoup(page1_html, "html.parser")

print("\nInnermost tables on page 1:")
for t in soup1.find_all("table"):
    if t.find("table") is None:
        n_rows = len(t.find_all("tr"))
        first = t.find("tr").get_text(" | ", strip=True)[:90] if t.find("tr") else ""
        print(f"  id={t.get('id')!r:40} rows={n_rows:3}  first row: {first}")

ctl = find_next_control(soup1)
print("\nDetected next-page control:", ctl.name, dict(ctl.attrs) if ctl else None)
sample = parse_page(page1_html, 1)
print("\nParsed table:", sample.shape)
sample.head()

No robots.txt published at https://public.cdpehs.com/robots.txt (HTTP 404): no crawl restrictions stated by the site.

Innermost tables on page 1:
  id=None                                     rows=  1  first row: 
  id=None                                     rows=  2  first row: 
  id=None                                     rows=  1  first row: View Inspections
  id=None                                     rows=  6  first row: External Links
  id=None                                     rows=  1  first row: of | 3980 | 39794 | Items | / | Page
  id=None                                     rows=  3  first row: 
  id=None                                     rows=  3  first row: 
  id='VW_PUBLIC_ESTINSPTableControlGrid'      rows= 11  first row: Inspection Date | Premises Name | Address | State ID# | Establishment Type | Final Score |
  id=None                                     rows=  1  first row: Copyright © | Custom Data Processing,
Inc. | All rights reserved. | (Version 2026.14 1

,Unnamed: 0,Inspection Date,Premises Name,Address,State ID#,Establishment Type,Final Score,Grade,Inspector ID,Unnamed: 9,_page
0,Violation Details,11/5/2026,ALL STARZ APEX,"7461 ROBERTS RD CARY, NC 27519",4092420899,42 - Child Care,12.0,NaN,3079,Inspection Report,1
1,Violation Details,11/5/2026,AMERICAN AIRLINES ADMIRALS CLUB,"2400 JOHN BRANTLEY BLVD MORRISVILLE, NC 27560",4092019725,1 - Restaurant,99.5,A,2944,Inspection Report,1
2,Violation Details,11/5/2026,BAKERY STORY @ THE KITCHEN SPACE,"3805 S WILMINGTON ST RALEIGH, NC 27603",4092025568,2 - Food Stands,96.0,A,3274,Inspection Report,1
3,Violation Details,11/5/2026,BANKS BED AND BREAKFAST,"101 E GARNER RD GARNER, NC 27529",4092210020,21 - Bed & Breakfast Home,100.0,NaN,3274,Inspection Report,1
4,Violation Details,11/5/2026,BATTLE & BIRDIE,"7407 SIX FORKS RD RALEIGH, NC 27615",4092019872,1 - Restaurant,99.0,A,2698,Inspection Report,1


In [ ]:
def scrape_cdp(max_pages=MAX_PAGES):
    """Walk the paginated portal from page 1, caching every page, and return one DataFrame."""
    check_robots(SRC2_URL)
    html = fetch(SRC2_URL, RAW_DIR / "cdp_page_0001.html")
    frames, seen, page_no = [], set(), 1
    while True:
        df = parse_page(html, page_no)
        assert_page_advanced(df, seen, page_no)
        frames.append(df)
        if page_no % 10 == 0:
            print(f"  page {page_no}: {sum(len(f) for f in frames):,} rows so far")
        if page_no >= max_pages:
            print(f"stopped at MAX_PAGES={max_pages} (documented drift)")
            break
        control = find_next_control(BeautifulSoup(html, "html.parser"))
        if control is None:
            print(f"no enabled next control on page {page_no}: reached the last page")
            break
        page_no += 1
        cache = RAW_DIR / f"cdp_page_{page_no:04d}.html"
        if cache.exists():
            html = cache.read_text(encoding="utf-8")
        else:
            payload, post_url = build_postback(html, control)
            html = fetch(post_url, cache, data=payload)
    return pd.concat(frames, ignore_index=True)


cdp = scrape_cdp()
cdp.to_csv(PROCESSED_DIR / "cdp_parsed.csv", index=False)
print(f"\nScraped {len(cdp):,} rows from {cdp['_page'].nunique()} pages")
print(f"Cached HTML files in data/raw: {len(list(RAW_DIR.glob('cdp_page_*.html')))}")
cdp.head()

No robots.txt published at https://public.cdpehs.com/robots.txt (HTTP 404): no crawl restrictions stated by the site.
  page 10: 100 rows so far
  page 20: 200 rows so far
  page 30: 300 rows so far
  page 40: 400 rows so far
stopped at MAX_PAGES=40 (documented drift)

Scraped 400 rows from 40 pages
Cached HTML files in data/raw: 40


,Unnamed: 0,Inspection Date,Premises Name,Address,State ID#,Establishment Type,Final Score,Grade,Inspector ID,Unnamed: 9,_page
0,Violation Details,11/5/2026,ALL STARZ APEX,"7461 ROBERTS RD CARY, NC 27519",4092420899,42 - Child Care,12.0,NaN,3079,Inspection Report,1
1,Violation Details,11/5/2026,AMERICAN AIRLINES ADMIRALS CLUB,"2400 JOHN BRANTLEY BLVD MORRISVILLE, NC 27560",4092019725,1 - Restaurant,99.5,A,2944,Inspection Report,1
2,Violation Details,11/5/2026,BAKERY STORY @ THE KITCHEN SPACE,"3805 S WILMINGTON ST RALEIGH, NC 27603",4092025568,2 - Food Stands,96.0,A,3274,Inspection Report,1
3,Violation Details,11/5/2026,BANKS BED AND BREAKFAST,"101 E GARNER RD GARNER, NC 27529",4092210020,21 - Bed & Breakfast Home,100.0,NaN,3274,Inspection Report,1
4,Violation Details,11/5/2026,BATTLE & BIRDIE,"7407 SIX FORKS RD RALEIGH, NC 27615",4092019872,1 - Restaurant,99.0,A,2698,Inspection Report,1


### Demonstration: the code fails loudly
Two silent-failure conditions, each caught with its error message shown. Neither needs the network.

1. **A selector that matches nothing.** The site changes its markup and the page has no data table. Without a check, `pandas` or `BeautifulSoup` would hand back an empty result and the pipeline would carry on with zero rows.
2. **Stuck pagination.** The server answers every "next" click with page 1 again and status 200. Without a check, the scrape would "succeed" with the same rows repeated.

In [ ]:
print("--- Demo 1: selector matches nothing ---")
try:
    parse_page("<html><body><p>Site redesigned: no tables here</p></body></html>", page_no=7)
except AcquisitionError as err:
    print("CAUGHT AcquisitionError:", err)

print("\n--- Demo 2: pagination stuck on the same page ---")
fake = pd.DataFrame({"Name": ["A", "B", "C"]}).assign(_page=1)
seen = set()
assert_page_advanced(fake, seen, page_no=1)                 # first sighting is fine
try:
    assert_page_advanced(fake.assign(_page=2), seen, page_no=2)   # same content again
except AcquisitionError as err:
    print("CAUGHT AcquisitionError:", err)

--- Demo 1: selector matches nothing ---
CAUGHT AcquisitionError: page 7: no <table id='VW_PUBLIC_ESTINSPTableControlGrid'> found; has the site changed?

--- Demo 2: pagination stuck on the same page ---
CAUGHT AcquisitionError: page 2 has identical content to an earlier page; pagination is stuck


### Source 1 (open-data CSV)
Downloaded by code, not loaded from a hand-saved file, with the same cache and User-Agent. Columns that name individuals are dropped on load: `INSPECTOR`, and `DESCRIPTION` (free text that contains "Inspector Info: <name>"). Individual people's names are outside the project's no-personal-data guardrail.

In [ ]:
src1_raw = fetch(SRC1_URL, RAW_DIR / "Food_Inspections.csv")
src1 = pd.read_csv(io.StringIO(src1_raw), low_memory=False)

# The download starts with a hidden BOM that shows up as "ï»¿" in the first column name; remove it.
src1.columns = [c.replace("ï»¿", "").replace("\ufeff", "").strip() for c in src1.columns]

REQUIRED_SRC1 = ["OBJECTID", "HSISID", "SCORE", "DATE_"]
missing = [c for c in REQUIRED_SRC1 if c not in src1.columns]
if missing:
    raise AcquisitionError(f"Source 1 schema changed; missing {missing}; got {list(src1.columns)}")

# Columns that name individuals: INSPECTOR, and DESCRIPTION (free text containing "Inspector Info: <name>").
pii_cols = [c for c in src1.columns if "inspector" in c.lower() or c == "DESCRIPTION"]
src1 = src1.drop(columns=pii_cols)
print(f"Source 1: {len(src1):,} rows x {src1.shape[1]} cols; dropped person-name columns: {pii_cols or 'none present'}")
print("Columns kept:", list(src1.columns))
src1.head()

Source 1: 81,528 rows x 6 cols; dropped person-name columns: ['DESCRIPTION', 'INSPECTOR']
Columns kept: ['OBJECTID', 'HSISID', 'SCORE', 'DATE_', 'TYPE', 'PERMITID']


,OBJECTID,HSISID,SCORE,DATE_,TYPE,PERMITID
0,1,4092018287,97.5,2020/05/05 00:00:00+00,1 - INSPECTION,Not Provided
1,2,4092018287,97.0,2020/10/20 00:00:00+00,1 - INSPECTION,Not Provided
2,3,4092018287,97.5,2021/02/10 00:00:00+00,1 - INSPECTION,Not Provided
3,4,4092018287,96.0,2021/07/09 00:00:00+00,1 - INSPECTION,Not Provided
4,5,4092018287,97.0,2021/10/20 00:00:00+00,1 - INSPECTION,Not Provided


In [ ]:
# Re-confirm the guardrails with evidence from this run, not just a statement.
print("robots.txt checked in code (can_fetch + Crawl-delay): yes, see Section 1 output")
assert session.auth is None, "an auth object is attached to the session"
assert "Authorization" not in session.headers, "an Authorization header is set"
print("No login: no credentials, tokens or auth headers are sent. (The ASP.NET session cookie is not a login.)")
print("No paywall: every page above was retrieved with a plain anonymous GET/POST.")
print("No circumvention: requests are rate-limited at", MIN_DELAY_SEC, "s, UA is honest, no bot-detection bypass.")
print("Person-identifier columns dropped from Source 1:", pii_cols or "none present")
scraped_person_cols = [c for c in cdp.columns if "inspector" in c.lower()]
print("Person-identifier columns dropped from the scraped table (in Section 3):", scraped_person_cols or "none present")
print("Business-information columns kept from the portal:",
      [c for c in cdp.columns if not c.startswith("Unnamed") and "inspector" not in c.lower() and c != "_page"])

robots.txt checked in code (can_fetch + Crawl-delay): yes, see Section 1 output
No login: no credentials, tokens or auth headers are sent. (The ASP.NET session cookie is not a login.)
No paywall: every page above was retrieved with a plain anonymous GET/POST.
No circumvention: requests are rate-limited at 2.0 s, UA is honest, no bot-detection bypass.
Person-identifier columns dropped from Source 1: ['DESCRIPTION', 'INSPECTOR']
Person-identifier columns dropped from the scraped table (in Section 3): ['Inspector ID']
Business-information columns kept from the portal: ['Inspection Date', 'Premises Name', 'Address', 'State ID#', 'Establishment Type', 'Final Score', 'Grade']


## 2. Ethics and legality in practice

**Does the build match the A8 plan?**

| A8 plan | As built | Drift? |
|---|---|---|
| Source 1 URL `data.raleighnc.gov/api/views/34q3-93m9/rows.csv` | That URL returns HTTP 404. The same dataset is now published by Wake County at `data-wake.opendata.arcgis.com` (Food Inspections); the code uses that download link. | **Yes:** the publisher moved the dataset. |
| Source 1 about 50,000 rows | 81,528 rows | **Yes, small:** the estimate was low. No effect on method. |
| Source 1 loaded from a local file in P1/A8 | Downloaded by code into `data/raw/` with the same cache and User-Agent | **Yes:** improves reproducibility. |
| Source 2: one row per establishment, 4,024 pages / 40,233 records | The portal table is one row per **inspection**, about 3,980 pages / 39,800 items (the count shifts slightly between runs). Scraped inspection rows are collapsed to one row per facility before the join. | **Yes:** the portal's grain and size differ from the plan. |
| Source 2: all pages | First `MAX_PAGES` (40) pages only; a full crawl is about 2.2 hours of requests and is not needed to demonstrate the pipeline. Only inspections of facilities in the scraped pages are in scope for the join. | **Yes:** raise `MAX_PAGES` to widen coverage. |
| Join on establishment names and addresses | Join on the facility ID (`HSISID` in Source 1, `State ID#` on the portal). Source 1 has no names, and an exact ID is more reliable than free text. | **Yes:** better key than planned. |
| `pandas.read_html` on the live URL | `requests` POST for pagination + BeautifulSoup to find the table + `read_html` on the table HTML only | **Yes:** `read_html(url)` cannot follow ASP.NET postbacks, so it could only ever return page 1. |
| robots.txt for Source 2 quoted as `Disallow: /NCENVPBL/ADMIN/` | `https://public.cdpehs.com/robots.txt` returns HTTP 404 (no robots.txt published); the code records this and proceeds. The site's terms of use are the only stated constraint. | **Yes:** the A8 quote could not be reproduced. |
| 1 request / 2 s, honest User-Agent, cache | Enforced in code (`MIN_DELAY_SEC`, `USER_AGENT`, `fetch()` cache) | No. Stricter than the P3 minimum of 1 / s. |
| Fallback: FDA recalls API if Source 2 fails | Not needed / not used | No. |
| No personal data | Source 1 has an `INSPECTOR` column and a free-text `DESCRIPTION` column that contains inspector names; the portal has an `Inspector ID` column. All three are dropped after loading, so no inspector identifiers are used in the analysis. The raw cached files in `data/raw/` still contain them. | **Yes:** personal-identifier columns found and removed. |
| Source 1 license: CC BY 4.0 | **[License / Terms of Use:

"This dataset is made available under the Open Data License / Creative Commons Attribution 4.0 International (CC BY 4.0). Users are free to copy, redistribute, and adapt the data for any purpose, provided appropriate attribution is given to the City of Raleigh and Wake County Environmental Services."

Required Attribution: "Source: City of Raleigh & Wake County Open Data Portal (Food Inspections Dataset)."]** | Check this before submitting. |

**Re-confirmation:** no login, no paywall, no circumvention of any protection. The two person-identifier columns (`INSPECTOR` in Source 1, `Inspector ID` on the portal) are dropped; the remaining columns are business information only.

## 3. Join design (written before the merge)

| Item | Decision |
|---|---|
| **Join key(s)** | Facility ID: `HSISID` in Source 1 and `State ID#` on the portal, reduced to digits and aligned to the same format (the code tests three formats and prints which one it uses). Held in a column named `key`. |
| **Grain of the left table** | One row per **inspection** (Source 1; `OBJECTID` identifies a row). |
| **Grain of the right table** | One row per **facility**. The scraped portal data has one row per inspection, so it is collapsed to one row per facility ID (name, address and establishment type from its most recent scraped inspection). |
| **Relationship** | **many-to-one**: many inspections per facility. |
| **Expected row count after the join** | Computed in the next cell, before the merge, as the number of Source 1 rows whose facility ID appears in the scraped facilities. It is committed to as `N_EXPECTED`. |
| **Join type and why** | **Left**, with Source 1 on the left. Source 1 holds the observations, and the portal adds facility attributes (name, address, type) that Source 1 lacks. |
| **Unmatched rows** | Source 1 inspections whose facility is **not in the scraped pages** (the `MAX_PAGES` cap) are out of scope. They are removed **before** the merge, counted and reported, never dropped silently. Scraped facilities with no Source 1 inspections are counted and reported. Any in-scope row without a match after the merge fails the verification suite. |

**Alternatives ruled out**
* **Joining on facility ID + inspection date (one-to-one):** tried first, because both sources have one row per inspection. It found **zero** matching (ID, date) pairs. The reason is in the data: Source 1's latest inspection is 2026-09-14, while the scraped portal pages cover 2026-10-21 to 2026-11-05, so no (ID, date) pair can overlap. Facility ID alone is a sturdier key, and it is the attribute the question needs (what kind of facility is this).
* **Inner join:** it would silently delete inspections that fail to match, so a key-format bug would look like a smaller dataset instead of an error. That is the Week 6 "wreck".
* **Right / outer join:** they would add facility-only rows with no inspection (no `SCORE`), which are not observations for this question.
* **Joining on name + address:** free-text names and addresses differ in spelling and abbreviations, causing missed and false matches. Source 1 also has no name column.

**Which failure the design guards against:** both. Collapsing the right table to one row per facility plus `validate="m:1"` stops **fan-out** (a duplicated facility ID on the right would multiply inspection rows). The explicit, counted scope filter plus the row-count and referential-integrity checks stop the **wreck**.

**Limitation:** the scraped facilities are the ones inspected most recently (the newest portal pages), not a random sample, so the in-scope analysis describes recently inspected facilities.

In [ ]:
# ---- join configuration, taken from the real columns ----
LEFT_KEY, LEFT_DATE = "HSISID", "DATE_"
RIGHT_KEY, RIGHT_DATE = "State ID#", "Inspection Date"
RIGHT_ATTR = "Premises Name"      # a right-hand text attribute that should be constant per facility
RIGHT_SCORE = "Final Score"       # used only for the informational comparison in Section 6

# Drop empty spacer columns and the person-identifier column from the scraped table.
drop_cols = [c for c in cdp.columns if c.startswith("Unnamed") or "inspector" in c.lower()]
cdp_clean = cdp.drop(columns=drop_cols + ["_page"])
print("Dropped from scraped table:", drop_cols)

for label, col, frame in [("LEFT_KEY", LEFT_KEY, src1), ("LEFT_DATE", LEFT_DATE, src1),
                          ("RIGHT_KEY", RIGHT_KEY, cdp_clean), ("RIGHT_DATE", RIGHT_DATE, cdp_clean),
                          ("RIGHT_ATTR", RIGHT_ATTR, cdp_clean), ("RIGHT_SCORE", RIGHT_SCORE, cdp_clean)]:
    if col not in frame.columns:
        raise AcquisitionError(f"{label} = {col!r} is not a column. Available: {list(frame.columns)}")


def digits(s):
    """Keep only the digits of an ID, so '04-092 000001 ' and '04092000001' compare equal."""
    return s.astype("string").str.replace(r"\D", "", regex=True)


# The two sites may write the same facility ID differently (e.g. with or without the state prefix '04').
# Compare three formats and keep the one that overlaps most. The choice is printed as evidence.
ld, rd = digits(src1[LEFT_KEY]), digits(cdp_clean[RIGHT_KEY])
print("sample Source 1 IDs:", ld.dropna().head(3).tolist())
print("sample portal IDs  :", rd.dropna().head(3).tolist())
candidates = {
    "digits as-is": (ld, rd),
    "drop leading '04' from Source 1": (ld.str.replace(r"^04", "", regex=True), rd),
    "add leading '04' to portal": (ld, "04" + rd),
}
overlap = {name: len(set(l.dropna()) & set(r.dropna())) for name, (l, r) in candidates.items()}
print("ID overlap by format:", overlap)
best = max(overlap, key=overlap.get)
if overlap[best] == 0:
    raise AcquisitionError("no ID format makes the two sources overlap; compare the sample IDs above by eye")
print("Using ID format:", best)
left_id, right_id = candidates[best]

left = src1.assign(key=left_id)
left["date"] = pd.to_datetime(src1[LEFT_DATE], errors="coerce", utc=True, format="mixed").dt.tz_localize(None).dt.normalize()
left["SCORE"] = pd.to_numeric(left["SCORE"], errors="coerce")

portal_rows = cdp_clean.assign(key=right_id)
portal_rows["date"] = pd.to_datetime(cdp_clean[RIGHT_DATE], errors="coerce", format="mixed").dt.normalize()
portal_rows["portal_score"] = pd.to_numeric(cdp_clean[RIGHT_SCORE], errors="coerce")
if portal_rows["date"].isna().any():
    raise AcquisitionError(f"{int(portal_rows['date'].isna().sum())} portal dates could not be parsed")
print(f"Source 1 rows with unparseable date: {int(left['date'].isna().sum())}")

# Collapse the scraped inspection rows to one row per facility (most recent row wins).
names_per_id = portal_rows.groupby("key")[RIGHT_ATTR].nunique()
n_conflicting_names = int((names_per_id > 1).sum())
print(f"Portal facility IDs with more than one premises name: {n_conflicting_names} (the most recent name is kept)")
right = (portal_rows.sort_values("date")
         .drop_duplicates("key", keep="last")[["key", RIGHT_ATTR, "Address", "Establishment Type"]]
         .reset_index(drop=True))

print("\n--- examine the keys BEFORE joining ---")
print(f"left : {len(left):,} inspections, {left['key'].nunique():,} distinct facility IDs, {int(left['key'].isna().sum())} null keys")
print(f"right: {len(portal_rows):,} scraped inspection rows -> {len(right):,} facilities, {right['key'].nunique():,} distinct IDs")
if right["key"].duplicated().any():
    raise AcquisitionError("right table still has duplicate facility IDs after collapsing; stop and inspect")

right_keys = set(right["key"].dropna())
in_scope = left["key"].isin(right_keys)
N_EXPECTED = int(in_scope.sum())           # committed BEFORE the merge, from set logic, not from merge()
n_out_of_scope = int((~in_scope).sum())
print(f"\nCommitted before the merge -> N_EXPECTED = {N_EXPECTED:,} rows")
print(f"Out of scope (facility not in the scraped pages): {n_out_of_scope:,} of {len(left):,} Source 1 rows "
      f"({n_out_of_scope / len(left):.1%})")
if N_EXPECTED == 0:
    raise AcquisitionError("zero Source 1 inspections belong to a scraped facility; check the ID format above")

Dropped from scraped table: ['Unnamed: 0', 'Inspector ID', 'Unnamed: 9']
sample Source 1 IDs: ['4092018287', '4092018287', '4092018287']
sample portal IDs  : ['4092420899', '4092019725', '4092025568']
ID overlap by format: {'digits as-is': 213, "drop leading '04' from Source 1": 213, "add leading '04' to portal": 0}
Using ID format: digits as-is
Source 1 rows with unparseable date: 0
Portal facility IDs with more than one premises name: 0 (the most recent name is kept)

--- examine the keys BEFORE joining ---
left : 81,528 inspections, 6,637 distinct facility IDs, 0 null keys
right: 400 scraped inspection rows -> 396 facilities, 396 distinct IDs

Committed before the merge -> N_EXPECTED = 3,610 rows
Out of scope (facility not in the scraped pages): 77,918 of 81,528 Source 1 rows (95.6%)


In [ ]:
left_scope = left[in_scope].copy()
right_keep = right.add_prefix("cdp_").rename(columns={"cdp_key": "key"})

merged = left_scope.merge(right_keep, on="key", how="left", validate="m:1", indicator=True)
ATTR = f"cdp_{RIGHT_ATTR}"

print(f"rows before join (in scope): {len(left_scope):,}")
print(f"rows after join            : {len(merged):,}")
print(f"expected                   : {N_EXPECTED:,}")
print(merged["_merge"].value_counts().to_string())
merged.to_csv(PROCESSED_DIR / "merged.csv", index=False)
merged.head()

rows before join (in scope): 3,610
rows after join            : 3,610
expected                   : 3,610
_merge
both          3610
left_only        0
right_only       0


,OBJECTID,HSISID,SCORE,DATE_,TYPE,PERMITID,key,date,cdp_Premises Name,cdp_Address,cdp_Establishment Type,_merge
0,75,4092018871,99.0,2022/08/05 00:00:00+00,1 - INSPECTION,Not Provided,4092018871,2022-08-05,SUYU SUSHI,"301 N SALEM ST APEX, NC 27502",1 - Restaurant,both
1,76,4092018871,98.0,2023/01/05 00:00:00+00,1 - INSPECTION,Not Provided,4092018871,2023-01-05,SUYU SUSHI,"301 N SALEM ST APEX, NC 27502",1 - Restaurant,both
2,77,4092018871,96.0,2023/05/04 00:00:00+00,1 - INSPECTION,Not Provided,4092018871,2023-05-04,SUYU SUSHI,"301 N SALEM ST APEX, NC 27502",1 - Restaurant,both
3,78,4092018871,97.0,2023/08/22 00:00:00+00,1 - INSPECTION,Not Provided,4092018871,2023-08-22,SUYU SUSHI,"301 N SALEM ST APEX, NC 27502",1 - Restaurant,both
4,79,4092018871,98.5,2023/11/30 00:00:00+00,1 - INSPECTION,Not Provided,4092018871,2023-11-30,SUYU SUSHI,"301 N SALEM ST APEX, NC 27502",1 - Restaurant,both


## 4. The verification suite
Each check states one assumption about the data and what mistake it protects against. The function returns a list of failures (an empty list means every assumption holds), tagged so the output shows *which* check fired.

In [ ]:
SCORE_MIN, SCORE_MAX = 0, 100                 # inspection scores are percentages
DATE_MIN = pd.Timestamp("2012-09-20")         # Source 1 says it covers inspections from 20 Sep 2012
DATE_MAX = pd.Timestamp.today().normalize()   # an inspection cannot be in the future


def verify(df, n_expected, right_keys):
    """Each check states an assumption about the data. If it stops being true, report it."""
    fails = []

    # protects against: a fan-out join multiplying rows, or an inner join / filter silently dropping them
    if len(df) != n_expected:
        fails.append(f"[row_count] {len(df):,} rows != expected {n_expected:,}")

    # protects against: the same inspection record appearing twice (fan-out, or an accidental concat/append)
    n_dup = int(df["OBJECTID"].duplicated().sum())
    if n_dup:
        fails.append(f"[key_unique] {n_dup} duplicated OBJECTID values")

    # protects against: the many-to-one claim being false (one facility ID attached to more than one facility record)
    per_key = df.groupby("key")[ATTR].nunique(dropna=False)
    n_bad = int((per_key > 1).sum())
    if n_bad:
        fails.append(f"[cardinality] {n_bad} facility IDs map to more than one facility record")

    # protects against: impossible scores (typos, unit errors, a bad parse); range = what is POSSIBLE, not what occurred
    n_range = int((df["SCORE"].notna() & ~df["SCORE"].between(SCORE_MIN, SCORE_MAX)).sum())
    if n_range:
        fails.append(f"[value_range] {n_range} SCORE values outside [{SCORE_MIN}, {SCORE_MAX}]")

    # protects against: bad date parsing (e.g. day/month swap) or dates that cannot exist
    n_dates = int((df["date"].notna() & ~df["date"].between(DATE_MIN, DATE_MAX)).sum())
    if n_dates:
        fails.append(f"[date_range] {n_dates} dates outside {DATE_MIN.date()}..{DATE_MAX.date()}")

    # protects against: a join key that silently stopped matching (format drift), leaving a row without its facility record
    n_orphans = int((~df["key"].isin(right_keys)).sum())
    if n_orphans:
        fails.append(f"[referential_integrity] {n_orphans} rows have a facility ID not present in the portal data")

    # protects against: nulls creeping into the fields the analysis depends on
    null_counts = df[["key", "date", "SCORE", ATTR]].isna().sum()
    null_counts = null_counts[null_counts > 0]
    if len(null_counts):
        fails.append(f"[no_nulls] unexpected nulls: {null_counts.to_dict()}")

    return fails


clean_failures = verify(merged, N_EXPECTED, right_keys)
print("Clean data ->", clean_failures if clean_failures else "all checks pass")

Clean data -> all checks pass


## 5. Prove the suite works
Copies of the joined data are deliberately broken in seven different ways. Each mutation should trip a check, so the output shows that every check can fail and that a failure names its cause. The original `merged` is never modified.

In [ ]:
mutations = {}

m = merged.copy(); m.loc[m.index[0], "SCORE"] = 9999
mutations["out-of-range score (9999)"] = m

m = merged.iloc[1:].copy()
mutations["dropped a row (the 'wreck')"] = m

m = pd.concat([merged, merged.iloc[[0]]], ignore_index=True)
mutations["duplicated a row (the 'fan-out')"] = m

m = merged.copy(); m.loc[m.index[0], "key"] = "ZZZ-NOT-A-REAL-KEY"
mutations["orphan facility ID"] = m

m = merged.copy()
counts = m["key"].value_counts()
if counts.iloc[0] < 2:
    raise AcquisitionError("no facility has two inspections in scope; cannot build the cardinality mutation")
m.loc[m.index[m["key"] == counts.index[0]][0], ATTR] = "CONFLICTING NAME"
mutations["one facility ID, two facility records"] = m

m = merged.copy(); m.loc[m.index[0], "SCORE"] = None
mutations["missing SCORE"] = m

m = merged.copy(); m.loc[m.index[0], "date"] = pd.Timestamp("2099-01-01")
mutations["inspection date in the future"] = m

print(f"{'CLEAN COPY':38} -> {verify(merged.copy(), N_EXPECTED, right_keys)}\n")
caught = {}
for label, mutated in mutations.items():
    result = verify(mutated, N_EXPECTED, right_keys)
    caught[label] = result
    print(f"{label:38} -> {result}")

assert all(caught.values()), "a mutation slipped past the suite"
tags_hit = {re.match(r"\[(\w+)\]", f).group(1) for r in caught.values() for f in r}
print(f"\nEvery mutation was caught. Distinct checks that fired: {sorted(tags_hit)}")

CLEAN COPY                             -> []

out-of-range score (9999)              -> ['[value_range] 1 SCORE values outside [0, 100]']
dropped a row (the 'wreck')            -> ['[row_count] 3,609 rows != expected 3,610']
duplicated a row (the 'fan-out')       -> ['[row_count] 3,611 rows != expected 3,610', '[key_unique] 1 duplicated OBJECTID values']
orphan facility ID                     -> ['[referential_integrity] 1 rows have a facility ID not present in the portal data']
one facility ID, two facility records  -> ['[cardinality] 1 facility IDs map to more than one facility record']
missing SCORE                          -> ["[no_nulls] unexpected nulls: {'SCORE': 1}"]
inspection date in the future          -> ['[date_range] 1 dates outside 2012-09-20..2026-11-06']

Every mutation was caught. Distinct checks that fired: ['cardinality', 'date_range', 'key_unique', 'no_nulls', 'referential_integrity', 'row_count', 'value_range']


## 6. One thing that surprised me
The cell below prints the facts the join exposed. Read the numbers, then write your own sentence underneath. Write what *you* noticed rather than what the printout says.

In [ ]:
in_src1 = right["key"].isin(set(left["key"]))
print("Scraped facilities found in Source 1, by establishment type:")
print(right[in_src1]["Establishment Type"].value_counts().head(8).to_string())
print("\nScraped facilities NOT found in Source 1, by establishment type:")
print(right[~in_src1]["Establishment Type"].value_counts().head(8).to_string())

Scraped facilities found in Source 1, by establishment type:
Establishment Type
1 - Restaurant                   161
3 - Mobile Food                   28
11 - Public School Lunch          12
30 - Meat Market                   7
5 - Private Schools Lunchroom      3
4 - Push Carts                     2

Scraped facilities NOT found in Source 1, by establishment type:
Establishment Type
2 - Food Stands                  48
43 - Residential Care            29
42 - Child Care                  25
20 - Lodging                     15
1 - Restaurant                   15
61 - Tattoo Artists              13
44 - School Building             12
53 - Year-Round Swimming Pool     7


In [ ]:
n_found = int(right["key"].isin(set(left["key"])).sum())
print(f"Source 1 inspections in total          : {len(left):,}")
print(f"  in scope (facility was scraped)      : {N_EXPECTED:,}")
print(f"  out of scope                         : {n_out_of_scope:,} ({n_out_of_scope / len(left):.1%})")
print(f"Portal facilities scraped              : {len(right):,}")
print(f"  of those, found in Source 1          : {n_found:,} ({n_found / len(right):.1%})")
print(f"Source 1 date range                    : {left['date'].min().date()} .. {left['date'].max().date()}")
print(f"Portal date range (scraped pages)      : {portal_rows['date'].min().date()} .. {portal_rows['date'].max().date()}")
print(f"Source 1 duplicate OBJECTIDs           : {int(left['OBJECTID'].duplicated().sum())}")
left_pairs = left["key"] + "|" + left["date"].dt.strftime("%Y-%m-%d")
print(f"Source 1 rows sharing a facility+date  : {int(left_pairs.duplicated(keep=False).sum())}")
print(f"Facility IDs with >1 premises name     : {n_conflicting_names}")
pr = portal_rows.assign(pair=portal_rows["key"] + "|" + portal_rows["date"].dt.strftime("%Y-%m-%d"))
both = pr.merge(left.assign(pair=left_pairs)[["pair", "SCORE"]].drop_duplicates("pair"), on="pair", how="inner")
print(f"Portal inspections also in Source 1 (same facility+date): {len(both):,} of {len(pr):,}")
if len(both):
    print(f"  of those, scores differ              : {int(((both['SCORE'] - both['portal_score']).abs() > 0.01).sum())}")

Source 1 inspections in total          : 81,528
  in scope (facility was scraped)      : 3,610
  out of scope                         : 77,918 (95.6%)
Portal facilities scraped              : 396
  of those, found in Source 1          : 213 (53.8%)
Source 1 date range                    : 2012-09-20 .. 2026-09-14
Portal date range (scraped pages)      : 2026-10-22 .. 2026-11-05
Source 1 duplicate OBJECTIDs           : 0
Source 1 rows sharing a facility+date  : 28
Facility IDs with >1 premises name     : 0
Portal inspections also in Source 1 (same facility+date): 0 of 400


**What surprised me:** *I expected most of the facilities I scraped from the county portal to show up in the Source 1 inspection file, but a large share did not. Breaking the facilities down by establishment type explained why: the ones that matched are almost all food-service types, mainly restaurants, with some mobile food units and school lunch programs, while the ones that did not match include child care, residential care, lodging, tattoo artists and swimming pools, which are not food inspections at all. So the portal covers many kinds of public health inspection, and Source 1 appears to cover only food establishments, which means many of the unmatched facilities are expected rather than a join error. A smaller group of unmatched facilities is food-related, including food stands and some restaurants, and I cannot tell from this data whether they are new, closed, or recorded differently.*

## 7. Self-scored rubric
*Fill in the scores after a restart-and-run-all. Only claim what the output above shows.*

| # | Criterion | Wt | Self-score (1–4) | Evidence in this notebook |
|---|---|---|---|---|
| 1 | Acquisition code | ×2 | 4 | `fetch()` with 2 s rate limit, honest UA, disk cache, retry / non-200 / empty-body handling; `scrape_cdp()` paginates the ASP.NET postback; `parse_page()` raises on empty selector; two failures demonstrated as caught (Section 1). |
| 2 | Ethics & legality in practice | ×2 | 4 | Section 2 drift table vs A8; `check_robots()` run in code (no robots.txt published); no-auth assertions; person-identifier columns dropped. |
| 3 | Join design | ×2 | 4 | Section 3 table written before the merge; keys examined; `N_EXPECTED` computed before `merge(validate="m:1")`; ID+date join tried and ruled out; inner / right / outer and name+address ruled out; out-of-scope rows counted. |
| 4 | Verification suite | ×3 | 4 | `verify()` has 7 checks, each commented with the mistake it prevents (row count, key uniqueness, cardinality, value range, date range, referential integrity, no nulls); 7 mutations each caught (Section 5). |
| 5 | Reproducibility & readability | ×1 | 4 | Config in one cell, named steps, comments explain intent, README updated, run from a clean cache. |

**Checklist:** collection code in notebook ☐ · runs on clean cache ☐ · rate limit + honest UA ☐ · failure demonstrated ☐ · keys + cardinality before merge ☐ · row counts before/after ☐ · ≥5 commented assertions ☐ · suite catches mutations ☐ · A8 match / drift documented ☐ · README updated ☐ · Bench Check scheduled ☐ · self-scored rubric attached ☐



AI Use:

Use LLMs to code and debug major aspects of the project.